In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error




import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:
# Load the dataset
data_path = os.path.join(path, 'Q1_data.csv')
Q1_data = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
print(f"Dataset shape: {Q1_data.shape}")
Q1_data.head()

In [ ]:
# Task 3: Write your code here:
Q1_data.info()

In [ ]:
# Task 4: Write your code here:
Q1_data.describe()

In [ ]:
# Task 5: Write your code here:
# DeliveryTime distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(Q1_data['Delivery_Time'].dropna(), bins=50, edgecolor='black', color = 'pink')
plt.title('DeliveryTime Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
# Drop the 'Order_ID' column from the data
Q1_data = Q1_data.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
# Handle missing values appropriately
print("Missing values:")
print(Q1_data.isnull().sum())

In [ ]:
# Fill categorical with mode
cat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day']
for col in cat_cols:
    Q1_data[col] = Q1_data[col].fillna(Q1_data[col].mode()[0])

# Fill numeric with median
num_cols = ['Courier_Experience_yrs']
for col in num_cols:
    Q1_data[col] = Q1_data[col].fillna(Q1_data[col].median())

# Remove rows where Delivery_Time is missing
Q1_data = Q1_data.dropna(subset=['Delivery_Time'])



In [ ]:
# Task 3: Write your code here:

duplicates = Q1_data.duplicated().sum() #Check and remove duplicates
print(f"Duplicates found: {duplicates}")

Q1_data = Q1_data.drop_duplicates() # Remove

In [ ]:
# Task 4: Write your code here:

cat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type'] #Encode categorical variables using One Hot Encoding
Q1_data_encoded = pd.get_dummies(Q1_data, columns=cat_cols, drop_first=True)

In [ ]:
# Task 5: Write your code here:

X = Q1_data_encoded.drop('Delivery_Time', axis=1) # Apply feature scaling for all features (Use StandardScaler)
y = Q1_data_encoded['Delivery_Time']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Task 6: Write your code here: #Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

In [ ]:
# Task 1: Write your code here:
X = pd.DataFrame(X_scaled, columns=Q1_data_encoded.drop('Delivery_Time', axis=1).columns)
y = Q1_data_encoded['Delivery_Time']

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# Task 2,3,4,5: Write your code here:


from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X):
    X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
    y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train and predict
    model = RandomForestRegressor(
        n_estimators=150,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:

# Refit RandomForest on the full data for feature importance
rf_full = RandomForestRegressor( n_estimators=200, random_state=42, n_jobs=-1)

rf_full.fit(X, y)

# Get feature importances
feature_importances = rf_full.feature_importances_
feature_names = Q1_data_encoded.drop('Delivery_Time', axis=1).columns

# Plot
plt.figure(figsize=(10, 8))
sorted_idx = np.argsort(feature_importances)

plt.barh(
    np.array(feature_names)[sorted_idx],
    feature_importances[sorted_idx]
)
plt.title("Feature Importance (RandomForest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Write your code here:

# Predict delivery time on the full dataset
y_pred_full = rf_full.predict(X)

plt.figure(figsize=(10, 5))
plt.hist(y_pred_full, bins=50, edgecolor='black')
plt.title("Predicted Delivery Time Distribution")
plt.xlabel("Predicted Delivery_Time (minutes)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Task Bonus: Write your code here: ## Some codes from my local lab
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores_ensemble = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Model 1: Random Forest
    rf = RandomForestRegressor(n_estimators=150, max_depth=20, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred_rf = rf.predict(X_val)

    # Model 2: Gradient Boosting
    gb = GradientBoostingRegressor( n_estimators=300, learning_rate=0.05, max_depth=3,  random_state=42)
    gb.fit(X_train, y_train)
    pred_gb = gb.predict(X_val)

    # Averaging predictions
    pred_avg = (pred_rf + pred_gb) / 2

    mae = mean_absolute_error(y_val, pred_avg)
    mae_scores_ensemble.append(mae)

print("Ensemble MAE for each fold:", mae_scores_ensemble)
print(f"Average Ensemble MAE: {np.mean(mae_scores_ensemble):.4f}")